[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Diseño de herramientas e ingeniería de contexto.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 3: *Herramientas, contexto y RAG*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se estudiará lo que más influye en la calidad de un agente: qué información entra en su ventana de contexto y cómo están diseñadas sus herramientas (nombres, descripciones, esquemas, granularidad y errores).

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a un modelo de lenguaje: las respuestas varían entre ejecuciones y consumen tokens de tu clave.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (ejecución local), definirá el modelo del curso y verificará la clave de API.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if not os.environ.get("GOOGLE_API_KEY") and MODELO.startswith("google_genai"):
    raise RuntimeError("Falta GOOGLE_API_KEY: agrégala como secreto de Codespaces y reconstruye el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from typing import Literal

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages.utils import count_tokens_approximately

---
## 1. Ingeniería de contexto.

En cada paso, el modelo solo sabe lo que está en su **ventana de contexto**: *prompt* de sistema, descripciones de herramientas, historial y resultados de herramientas. La ingeniería de contexto decide **qué entra, en qué orden y cuándo se resume o se quita**.

| Fuente de contexto | Riesgo | Técnica |
|---|---|---|
| *Prompt* de sistema | Reglas contradictorias o vagas | Instrucciones cortas, verificables |
| Herramientas | Demasiadas, descripciones ambiguas | Pocas herramientas, bien descritas; selección dinámica (`08`) |
| Historial | Crece en cada paso | Resumen o recorte (`08`) |
| Resultados de herramientas | Documentos enteros, JSON gigantes | Devolver solo lo necesario; RAG (`06`) |

* La siguiente celda definirá las herramientas de políticas.

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

* La siguiente celda estimará cuántos tokens ocuparían **todas** las políticas si se metieran al contexto.

In [ ]:
texto_completo = "\n".join(leer_politica.invoke({"nombre": n}) for n in listar_politicas.invoke({}))
count_tokens_approximately([texto_completo])

**Observa:** con tres políticas cortas el número es pequeño; con cientos de documentos no cabrían, y aunque cupieran, el modelo rendiría peor. Por eso se recupera solo lo relevante (`06`).

---
## 2. Diseño de herramientas.

| Elemento | Mal | Bien |
|---|---|---|
| Nombre | `procesar`, `tool1` | `crear_ticket`, `buscar_politicas` |
| Descripción | "Hace cosas con tickets" | Qué hace, cuándo usarla y cuándo **no** |
| Argumentos | `datos: dict` | Tipos concretos, `Literal` para valores cerrados |
| Errores | Excepción con *traceback* | Mensaje que dice cómo corregir la llamada |
| Granularidad | Una herramienta que hace todo | Una acción por herramienta |
| Salida | El registro completo de la base de datos | Solo los campos que el modelo necesita |

* La siguiente celda definirá tres herramientas **mal diseñadas** para el ejercicio 1.

In [ ]:
@tool
def procesar(datos: dict) -> str:
    """Procesa datos."""
    return f"Procesado: {datos}"


@tool
def herramienta2(q: str) -> str:
    """Información."""
    return leer_politica.invoke({"nombre": q})


@tool
def gestionar(accion: str, x: str = "") -> str:
    """Gestiona tickets."""
    return f"{accion} {x}: hecho"

* La siguiente celda mostrará qué llamadas hace un agente con esas herramientas.

In [ ]:
agente_malo = create_agent(MODELO, tools=[procesar, herramienta2, gestionar],
                           system_prompt="Eres el asistente de soporte interno.")
r = agente_malo.invoke({"messages": [{"role": "user", "content": "¿Cuál es el tope de hotel nacional? Y ábreme un ticket porque mi monitor parpadea."}]})
for m in r["messages"]:
    if m.type == "ai" and m.tool_calls:
        print(m.tool_calls)

### Ejercicio 1 · Reescribir herramientas.

Reescribe las tres herramientas anteriores como `leer_politica` (ya existe), `crear_ticket` y `consultar_tickets`, con nombres, descripciones, tipos y errores útiles. Ejecuta la misma pregunta y compara las llamadas.

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2 · Predecir antes de ejecutar.

Antes de ejecutar tu versión, escribe qué herramientas esperas que llame el agente y en qué orden. Después compara con la traza. ¿Acertaste? ¿Qué te sorprendió?

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 3. Errores útiles y salidas acotadas.

Cuando una herramienta falla, el agente solo puede recuperarse si el error le dice **qué hacer**. Y cuando una herramienta devuelve demasiado texto, llena el contexto: conviene recortar.

* La siguiente celda mostrará el error útil de `leer_politica` con un nombre inexistente.

In [ ]:
leer_politica.invoke({"nombre": "viaticos"})

**💡 Ideas clave de la sección 3:**
* El nombre y la descripción de una herramienta son su *prompt*.
* Un error útil indica cómo corregir la llamada.
* Menos herramientas y salidas más cortas = menos confusión y menos tokens.

### Reto 1 · Proteger el contexto.

Escribe un decorador `recortar(max_caracteres)` que envuelva la función de una herramienta y recorte su salida, agregando al final `"[... recortado: N caracteres omitidos]"`. Aplícalo a una versión de `leer_politica` con 300 caracteres.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre.

**Práctica:** terminar el ejercicio 1 y comparar las trazas del agente malo y del bueno en *LangSmith*.

**Siguiente *notebook*:** `06_rag_como_herramienta.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>